In [1]:
from pathlib import Path
import pandas as pd

rna_dir = Path("../data/raw/rna")

rna_files = list(
    rna_dir.rglob("*.rna_seq.augmented_star_gene_counts.tsv")
)

print("RNA files:", len(rna_files))
rna_files[:3]

RNA files: 541


[WindowsPath('../data/raw/rna/000d81dd-9ba4-4852-9090-2bf22f6483f0/09755ce8-ed89-411a-a42f-b3edc4e41eeb.rna_seq.augmented_star_gene_counts.tsv'),
 WindowsPath('../data/raw/rna/00174492-1adc-4310-8147-a622befbb466/84e40c7e-2f60-4461-b14c-0859697295a4.rna_seq.augmented_star_gene_counts.tsv'),
 WindowsPath('../data/raw/rna/00ddbdd1-f29d-438b-820b-aacb111a3ec4/e635dbc7-ef8d-45e2-acd2-b961f0c8b600.rna_seq.augmented_star_gene_counts.tsv')]

In [2]:
sample_file = rna_files[0]

sample_rna = pd.read_csv(
    sample_file,
    sep="\t",
    comment="#"
)

print(sample_file)
print(sample_rna.shape)
print(sample_rna.columns.tolist())

sample_rna.head()

..\data\raw\rna\000d81dd-9ba4-4852-9090-2bf22f6483f0\09755ce8-ed89-411a-a42f-b3edc4e41eeb.rna_seq.augmented_star_gene_counts.tsv
(60664, 9)
['gene_id', 'gene_name', 'gene_type', 'unstranded', 'stranded_first', 'stranded_second', 'tpm_unstranded', 'fpkm_unstranded', 'fpkm_uq_unstranded']


,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
0,N_unmapped,NaN,NaN,1814468,1814468,1814468,NaN,NaN,NaN
1,N_multimapping,NaN,NaN,3294784,3294784,3294784,NaN,NaN,NaN
2,N_noFeature,NaN,NaN,4053713,22898782,22748008,NaN,NaN,NaN
3,N_ambiguous,NaN,NaN,3953132,915908,921288,NaN,NaN,NaN
4,ENSG00000000003.15,TSPAN6,protein_coding,2157,1097,1060,50.3093,13.6366,13.0426


In [3]:
for col in sample_rna.columns:
    print(col)

gene_id
gene_name
gene_type
unstranded
stranded_first
stranded_second
tpm_unstranded
fpkm_unstranded
fpkm_uq_unstranded


In [4]:
gene_rows = sample_rna[
    sample_rna["gene_id"].astype(str).str.startswith("ENSG")
].copy()

print("전체 행:", len(sample_rna))
print("실제 gene 행:", len(gene_rows))

gene_rows.head()

전체 행: 60664
실제 gene 행: 60660


,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded
4,ENSG00000000003.15,TSPAN6,protein_coding,2157,1097,1060,50.3093,13.6366,13.0426
5,ENSG00000000005.6,TNMD,protein_coding,26,14,12,1.8636,0.5051,0.4831
6,ENSG00000000419.13,DPM1,protein_coding,978,499,480,85.7241,23.2360,22.2239
7,ENSG00000000457.14,SCYL3,protein_coding,604,452,465,9.2839,2.5165,2.4068
8,ENSG00000000460.17,C1orf112,protein_coding,182,268,266,3.2253,0.8742,0.8362


In [5]:
gene_rows["gene_type"].value_counts().head(20)

gene_type
protein_coding                        19962
lncRNA                                16901
processed_pseudogene                  10167
unprocessed_pseudogene                 2614
misc_RNA                               2212
snRNA                                  1901
miRNA                                  1881
TEC                                    1057
snoRNA                                  943
transcribed_unprocessed_pseudogene      939
transcribed_processed_pseudogene        500
rRNA_pseudogene                         497
IG_V_pseudogene                         187
IG_V_gene                               145
transcribed_unitary_pseudogene          138
TR_V_gene                               106
unitary_pseudogene                       98
TR_J_gene                                79
scaRNA                                   49
polymorphic_pseudogene                   48
Name: count, dtype: int64

In [6]:
protein_coding = gene_rows[
    gene_rows["gene_type"] == "protein_coding"
].copy()

print("protein-coding genes:", len(protein_coding))

protein_coding[
    ["gene_id", "gene_name", "tpm_unstranded"]
].head()

protein-coding genes: 19962


,gene_id,gene_name,tpm_unstranded
4,ENSG00000000003.15,TSPAN6,50.3093
5,ENSG00000000005.6,TNMD,1.8636
6,ENSG00000000419.13,DPM1,85.7241
7,ENSG00000000457.14,SCYL3,9.2839
8,ENSG00000000460.17,C1orf112,3.2253


In [7]:
print("TPM 결측치:", protein_coding["tpm_unstranded"].isna().sum())
print("TPM 최소값:", protein_coding["tpm_unstranded"].min())
print("TPM 최대값:", protein_coding["tpm_unstranded"].max())

protein_coding["tpm_unstranded"].describe()

TPM 결측치: 0
TPM 최소값: 0.0
TPM 최대값: 38151.6508


count    19962.000000
mean        46.511838
std        465.693330
min          0.000000
25%          0.345325
50%          8.684450
75%         31.129075
max      38151.650800
Name: tpm_unstranded, dtype: float64

In [8]:
print("gene_name 결측:", protein_coding["gene_name"].isna().sum())
print("고유 gene_name:", protein_coding["gene_name"].nunique())
print("중복 gene_name:", protein_coding["gene_name"].duplicated().sum())

gene_name 결측: 0
고유 gene_name: 19938
중복 gene_name: 24


In [9]:
duplicated_genes = protein_coding[
    protein_coding["gene_name"].duplicated(keep=False)
].sort_values("gene_name")

duplicated_genes[
    ["gene_id", "gene_name", "tpm_unstranded"]
]

,gene_id,gene_name,tpm_unstranded
15283,ENSG00000182584.6,ACTL10,0.0522
60647,ENSG00000288649.1,ACTL10,0.0000
17595,ENSG00000197976.12,AKAP17A,42.6825
17596,ENSG00000197976.12_PAR_Y,AKAP17A,0.0000
17108,ENSG00000196433.13,ASMT,0.2959
17109,ENSG00000196433.13_PAR_Y,ASMT,0.0000
12516,ENSG00000169093.16,ASMTL,37.0559
12517,ENSG00000169093.16_PAR_Y,ASMTL,0.0000
27,ENSG00000002586.20,CD99,94.0583
28,ENSG00000002586.20_PAR_Y,CD99,0.0000


In [12]:
print("gene_id 고유 개수:", protein_coding["gene_id"].nunique())
print("gene_id 중복:", protein_coding["gene_id"].duplicated().sum())

gene_id 고유 개수: 19962
gene_id 중복: 0


In [13]:
n_pc = len(protein_coding)
n_unique_symbol = protein_coding["gene_name"].nunique()
n_dup_rows = protein_coding["gene_name"].duplicated().sum()

print("Protein-coding rows:", n_pc)
print("Unique gene symbols:", n_unique_symbol)
print("Duplicated symbol rows:", n_dup_rows)

Protein-coding rows: 19962
Unique gene symbols: 19938
Duplicated symbol rows: 24


### Gene annotation audit

- Protein-coding gene rows: 19,962
- Unique gene symbols: 19,938
- Missing gene symbols: 0
- Duplicated gene-symbol rows: 24
- Duplicates include PAR_Y annotations and multiple Ensembl IDs mapped to the same symbol.
- Gene-level modeling will retain Ensembl gene IDs.
- Pathway-level analysis will collapse duplicated gene symbols using a predefined aggregation rule.